# NewsBot 2.0 · Notebook 06 – Conversational Interface
**ITAI 2373 Final Project · Mourinho Hernanto**

**Goal:** understand natural-language questions (intent + slots), keep conversation context for follow-ups, and answer from the analysed news.

**Course modules:** 12 (conversational AI), 7 (intent classification), 8 (entities)

> Runs top-to-bottom in Google Colab (first cell installs everything). All logic lives in `src/` –
> this notebook demonstrates and evaluates it.

In [1]:
# ⚙️ Setup – works in Google Colab (clones the repo + installs) and locally
import os, sys, subprocess
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
ROOT = Path("/content/ITAI2373-NewsBot-Final/ITAI2373-NewsBot-Final") if IN_COLAB else Path.cwd().resolve().parent
if IN_COLAB and not ROOT.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/Haryapandhita/ITAI2373-NewsBot-Final.git",
                    "/content/ITAI2373-NewsBot-Final"], check=True)
sys.path.insert(0, str(ROOT))
from scripts.colab_setup import setup
ROOT = setup()

import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 120)
# Transformer models (summarisation, translation, multilingual embeddings) are used when
# available (Colab: Runtime -> Change runtime type -> T4 GPU recommended); otherwise
# NewsBot falls back to classical methods automatically.
USE_TRANSFORMERS = os.environ.get("NEWSBOT_USE_TRANSFORMERS", "1") == "1"
print("Transformers enabled:", USE_TRANSFORMERS)

Project root: /home/claude/NewsBot-Final/ITAI2373-NewsBot-Final  |  Colab: False


Transformers enabled: True


## 1. Intent classification

In [2]:
from src.conversation.intent_classifier import IntentClassifier, generate_training_data, HELD_OUT_QUERIES
X, y = generate_training_data()
print(len(X), "generated training queries over", len(set(y)), "intents"); print(pd.Series(y).value_counts().to_dict())
ic = IntentClassifier().train(X, y)
ev = ic.evaluate()
print(f"Held-out hand-written queries: accuracy {ev['accuracy']:.3f}, macro-F1 {ev['macro_f1']:.3f} (n={ev['n']})")
ev["errors"]

960 generated training queries over 13 intents
{'search': 126, 'summarize': 108, 'sentiment': 102, 'compare': 78, 'insights': 78, 'topics': 66, 'trends': 66, 'greeting': 66, 'entity': 60, 'classify': 60, 'export': 54, 'translate': 48, 'help': 48}
Held-out hand-written queries: accuracy 0.977, macro-F1 0.983 (n=43)


[('Sum up what happened with Apple', 'summarize', np.str_('search'))]

## 2. Slot extraction (category, sentiment, time, language, entities, format …)

In [3]:
from src.conversation.query_processor import QueryProcessor
qp = QueryProcessor(ic, reference_date=pd.Timestamp("2026-11-13", tz="UTC"))
for q in ["Show me positive tech news from this week", "Compare Spanish and German coverage of the economy",
          "Top 3 stories about Elon Musk in Q3 2025", "Export the results as csv"]:
    p = qp.parse(q)[0]; print(f"{q}\n   → {p.intent} ({p.confidence}) {p.slots}\n")

Show me positive tech news from this week
   → search (0.789) {'category': ['tech'], 'sentiment': 'positive', 'date_from': '2026-11-06', 'date_to': '2026-11-13'}

Compare Spanish and German coverage of the economy
   → compare (0.89) {'category': ['business'], 'language': ['es', 'de']}

Top 3 stories about Elon Musk in Q3 2025
   → search (0.873) {'date_from': '2025-07-01', 'date_to': '2025-09-30', 'top_k': 3, 'entities': ['Elon Musk']}

Export the results as csv
   → export (0.99) {'format': 'csv'}



## 3. The full chatbot (NewsBot2) – with context-aware follow-ups

In [4]:
from src.newsbot import NewsBot2
bot = NewsBot2(use_transformers=USE_TRANSFORMERS).build()
def ask(q):
    print("🧑", q); print("🤖", bot.chat(q)); print("-" * 100)

20:53:59 | INFO    | newsbot | load data                       0.1s


20:54:00 | INFO    | newsbot | classifier                      0.6s


20:54:00 | INFO    | newsbot | language detector               0.1s


20:54:00 | INFO    | newsbot | enrich corpus                   0.2s


20:55:55 | INFO    | src.analysis.ner_extractor | Knowledge graph: 10744 nodes, 34403 edges


20:55:55 | INFO    | newsbot | knowledge graph               114.2s


... (4 outputs left out)

20:57:21 | INFO    | src.language_models.embeddings | Indexed 15242 articles with lsa backend


20:57:21 | INFO    | newsbot | semantic index                 68.2s


20:57:21 | INFO    | newsbot | intent classifier               0.1s


20:57:21 | INFO    | newsbot | NewsBot 2.0 ready: 15242 articles, 6 languages


In [5]:
for q in ["hi", "Show me positive tech news from this year", "what about last year?", "only negative ones", "summarize them"]:
    ask(q)

🧑 hi
🤖 Hi! I'm NewsBot 2.0. Ask me about news topics, sentiment, entities or trends (type 'help' for ideas).
----------------------------------------------------------------------------------------------------
🧑 Show me positive tech news from this year


🤖 These articles match for positive tech news from 2026-01-01 to 2026-11-13 (5 shown):
  1. Latest news bulletin | August 13th, 2026 - Morning  [tech · en · 2026-08-13 · positive]
  2. German futurists: Openness and fast iteration drive China's tech  [tech · en · 2026-08-13 · positive]
  3. Life science, industrials, financials and tech as key AI beneficiaries  [tech · en · 2026-08-13 · positive]
  4. Inzi Controls wins cell connection system order from Hyundai Mobis  [tech · en · 2026-08-13 · positive]
  5. Apple explores publisher deals to power smarter Siri  [tech · en · 2026-08-13 · positive]
You can ask me to summarize them, compare them, or show only positive/negative ones.
----------------------------------------------------------------------------------------------------
🧑 what about last year?


20:57:22 | WARNING | src.multilingual.translator | Translation model Helsinki-NLP/opus-mt-de-en unavailable (No module named 'transformers')


🤖 These articles match for positive tech news from 2025-01-01 to 2025-12-31 (5 shown):
  1. Cramer Called Tech's Bounce Back a Stay of Execution, But Amazon (AMZN), Apple (AAPL) and Broadcom (  [tech · en · 2025-04-02 · positive]
  2. Sinclair Breaks Ground as First Broadcaster Approved for Drone Operations Over People  [tech · en · 2025-04-02 · positive]
  3. Students at Thomasville High dive into cybersecurity  [tech · en · 2025-04-02 · positive]
  4. Another DJI rival bites the dust - Autel quits consumer drones, here's what you need to know  [tech · en · 2025-08-21 · positive]
  5. Shenmue Is The Most Influential Game Of All Time According To BAFTA  [tech · en · 2025-04-02 · positive]
You can ask me to summarize them, compare them, or show only positive/negative ones.
----------------------------------------------------------------------------------------------------
🧑 only negative ones
🤖 Here's what I found for negative tech news from 2025-01-01 to 2025-12-31 (5 shown):
  1. Doge

In [6]:
for q in ["Compare Apple and Google", "How are Donald Trump and China connected?", "Who is Elon Musk?"]:
    ask(q)

🧑 Compare Apple and Google


🤖 Comparison:
  • Google: 704 articles, average tone +0.06, 48% positive, mostly tech
  • Apple: 423 articles, average tone +0.10, 60% positive, mostly tech
Google gets more coverage; Apple gets the friendlier tone.
----------------------------------------------------------------------------------------------------
🧑 How are Donald Trump and China connected?
🤖 Donald Trump and China appear together in 19 articles; extracted relations: get (1x), back (1x).
----------------------------------------------------------------------------------------------------
🧑 Who is Elon Musk?
🤖 Elon Musk (PERSON) is mentioned in 34 articles, mostly in tech (22), business (6), politics (6). Most often mentioned with: Donald Trump, United States, AI, Twitter, Tesla.
----------------------------------------------------------------------------------------------------


In [7]:
for q in ["What topics are trending in business?", "How has sentiment about China changed over time?",
          "Is coverage of the economy mostly negative?", "Compare Spanish and German politics news"]:
    ask(q)

🧑 What topics are trending in business?


🤖 Main topics:
  • stock / price / market – 18% of articles (stock, price, market, trading, investors, shares, stocks, co...)
  • growth / economy / economic – 16% of articles (growth, economy, economic, prices, bank, rate, market, sales...)
  • ai / technology / solutions – 15% of articles (ai, technology, solutions, development, industry, digital, b...)
  • users / features / online – 11% of articles (users, features, online, up, using, google, phone, mobile, f...)
  • court / law / state – 9% of articles (court, law, state, case, federal, justice, legal, attorney, ...)
  • health / treatment / study – 6% of articles (health, treatment, study, patients, research, cancer, care, ...)
Growing fastest: stock / price / market, game / season / team, match / cup / win
----------------------------------------------------------------------------------------------------
🧑 How has sentiment about China changed over time?


🤖 Trend for news about China: coverage became more negative (2023Q4: +0.12 → 2026Q4: +0.09).
Busiest period: 2023Q4 with 165 articles.
  2025Q1: +0.10 (10 articles)
  2025Q2: -0.01 (41 articles)
  2025Q3: -0.01 (43 articles)
  2025Q4: +0.13 (2 articles)
  2026Q1: -0.02 (37 articles)
  2026Q2: +0.03 (7 articles)
  2026Q3: +0.12 (101 articles)
  2026Q4: +0.09 (59 articles)
----------------------------------------------------------------------------------------------------
🧑 Is coverage of the economy mostly negative?
🤖 Sentiment overview: coverage of business news is mostly positive (average score +0.10 across 2677 articles; 58% positive, 20% negative). The strongest emotion is trust.
----------------------------------------------------------------------------------------------------
🧑 Compare Spanish and German politics news
🤖 Comparison:
  • Spanish: 1171 articles, average tone -0.02, 15% positive, mostly politics
  • German: 61 articles, average tone -0.24, 0% positive, mostly politic

In [8]:
ask("show German business news and then summarize them")    # multi-part question
ask("Classify this article: Shares in Tesla fell 5% after the earnings call disappointed investors on Wall Street.")
ask("give me an overview"); ask("export the results as csv"); ask("help")

🧑 show German business news and then summarize them


🤖 These articles match for business news in de (5 shown):
  1. Ein Befreiungsschlag mit Fragezeichen  [business · de · 2026-01-15 · negative]
  2. Ark Invest kauft 3IQ Solana Staking ETF-Anteile für 51.000 $ | Phemex News  [business · de · 2026-09-24 · positive]
  3. Spot-Gold durchbricht 4.300 US-Dollar pro Unze | Phemex News  [business · de · 2026-09-24 · negative]
  4. Wal kauft 536 BTC, häuft 2.460 BTC in 20 Tagen an | Phemex News  [business · de · 2026-09-24 · negative]
  5. Tesla in Europa mit Absatzminus - Sinkende Auslastung in Grünheide bedroht Gigafactory - Pravda Deut  [business · de · 2026-01-15 · negative]
You can ask me to summarize them, compare them, or show only positive/negative ones.

In a nutshell (5 articles on business news in de):
Der Ausbruch markiert einen bedeutenden technischen Meilenstein für die Goldmärkte, wobei die Preise neue Hochs über der Schwelle von 4.300 US-Dollar etablieren. Ark Invest von Cathie Wood hat 5.589 Anteile des 3IQ Solana Staking ETF er

## 4. Optional: polishing answers with a local LLM (Ollama)
Set `NEWSBOT_USE_LLM=1` (and run `ollama serve` with e.g. `llama3.2`) to let a local LLM rephrase the **grounded** draft.
The LLM receives only the facts NewsBot computed and is told not to add any; without Ollama the template answer is used.

In [9]:
from src.language_models.generator import OllamaClient
llm = OllamaClient()
print("Ollama available:", llm.available())
draft = bot.chat("Who is Elon Musk?")
print(llm.rewrite(draft) if llm.available() else draft)

Ollama available: False
Elon Musk (PERSON) is mentioned in 34 articles, mostly in tech (22), business (6), politics (6). Most often mentioned with: Donald Trump, United States, AI, Twitter, Tesla.


## Key takeaways
* Template-bootstrapped intent model reaches high accuracy on differently-worded held-out questions.
* Rule + NER slot filling handles categories, sentiment, relative/absolute dates, languages, entities and formats.
* Conversation memory enables follow-ups ("what about last year?", "only negative ones", "summarize them") and
  multi-part questions; answers are grounded in the analysed corpus.